## Assignment 1 
### Part B

In [1]:

# Our certainty equivalence 
erik_CE = {
    "X" : 0.10,
    "Y" : 0.12,
    "Z" : 0.05
}

elaine_CE = {
    "X" : 0.04,
    "Y" : 0.03,
    "Z" : 0.035
}

chen_CE = {
    "X" : 0.07,
    "Y" : 0.05,
    "Z" : 0.02
}

# stock fund
E_rs = 0.1
std_rs = 0.18

# bond fund
E_bf = 0.06
std_bf = 0.1

# correlation factor between stock and bonds
rho_sb = 0.25

# risk-free rate
rf = 0.03

## Question 1

In [2]:
import math
import pandas as pd
# Expected return and standard deviation of each bet
bet_X = { #outcome and probability
    0.3: 0.5,
    -0.1: 0.5
}

bet_Y = {
    0.15: 0.8,
    -0.25: 0.2
}

bet_Z = {
    0.39: 0.2,
    -0.01: 0.8
}

e_val = {}
std_val = {}
bets = {
    "X": bet_X, 
    "Y": bet_Y, 
    "Z": bet_Z
    }
for name, bet in bets.items():
    mean = sum(payoff * prob for payoff, prob in bet.items())
    std = math.sqrt(
        sum((payoff - mean) ** 2 * prob for payoff, prob in bet.items())
    )
    e_val[name] = mean
    std_val[name] = std

print("Expected Values (%):", [f"{v:.2%}" for v in e_val.values()])
print("Standard deviations (%):", [f"{v:.2%}" for v in std_val.values()])

def risk_aversion(e_val, std_val, ce):
    A = 2 * (e_val - ce) / std_val ** 2
    return A

members = {
    "erik": erik_CE,
    "elaine": elaine_CE,
    "chen": chen_CE
}
risk_aversion_df = pd.DataFrame({
    m: {b: risk_aversion(e_val[b], std_val[b], ces[b]) for b in bets}
    for m, ces in members.items()
})  # rows = bets, columns = members

print(risk_aversion_df.round(2))

Expected Values (%): ['10.00%', '7.00%', '7.00%']
Standard deviations (%): ['20.00%', '16.00%', '16.00%']
   erik  elaine  chen
X -0.00    3.00  1.50
Y -3.91    3.12  1.56
Z  1.56    2.73  3.91


## Question 2
### The Risky Portfolio
The weights of Stock Fund ($S$) and Bond Fund ($B$) in the Sharpe-maximising portfolio $P^*$ are given by:

$$w_S^* = \frac{R_S \sigma_B^2 - R_B \sigma_{SB}}{R_S \sigma_B^2 + R_B \sigma_S^2 - (R_S + R_B) \sigma_{SB}}$$

where $R_S = E[r_S] - r_f$ and $R_B = E[r_B] - r_f$ are the excess returns and $\sigma_{SB} = \rho_{SB}\sigma_S\sigma_B$.

$$w_B^* = 1 - w_S^*$$

The weights are independent of the investor's risk preference. This is because it is the mean-variance optimal mix of two risky assets when we consider their expected return and their covariance matrix. 
The capital allocation line has constant Sharpe ratio. Different points represent different combinations of the risk-asset and the risky portfolio. Where an investor sits on the line depends on their risk preference
### Optimal Capital Allocation to the Risky Portfolio ($y^*$)

The fraction of wealth allocated to the optimal risky portfolio, denoted as $y^*$, is derived by maximizing the investor's mean-variance utility function $U(y)$ with respect to $y$:

$$U(y) = E[r_C] - \frac{1}{2} A \sigma_C^2$$

Substituting the expected return $E[r_C] = r_f + y(E[r_{P^*}] - r_f)$ and variance $\sigma_C^2 = y^2 \sigma_{P^*}^2$ of the complete portfolio into the utility function yields:

$$U(y) = r_f + y\left(E[r_{P^*}] - r_f\right) - \frac{1}{2} A y^2 \sigma_{P^*}^2$$

To locate the utility-maximizing allocation, we evaluate the **first-order condition (FOC)** by setting the first derivative of $U(y)$ with respect to $y$ to zero:

$$\frac{dU}{dy} = \left(E[r_{P^*}] - r_f\right) - A y \sigma_{P^*}^2 = 0$$

Solving for $y$ gives the optimal allocation formula:

$$y^* = \frac{E[r_{P^*}] - r_f}{A \sigma_{P^*}^2}$$

In [3]:
var_rs = std_rs ** 2
var_bf = std_bf ** 2
cov_sb = rho_sb * std_rs * std_bf
ex_rs = E_rs - rf
ex_bf = E_bf - rf
w_rs = (ex_rs * var_bf - ex_bf * cov_sb) / (
    ex_rs * var_bf + ex_bf * var_rs - (ex_rs + ex_bf) * cov_sb
)
w_bf = 1 - w_rs

E_rp = w_rs * E_rs + w_bf * E_bf
var_rp = (w_rs ** 2) * var_rs + (w_bf ** 2) * var_bf \
    + 2 * w_rs * w_bf * cov_sb
std_rp = math.sqrt(var_rp)
sharpe_rp = (E_rp - rf) / std_rp

print("Optimal Risky Portfolio (P*):")
print(f"  w_S*: {w_rs:.4f} ({w_rs:.2%})")
print(f"  w_B*: {w_bf:.4f} ({w_bf:.2%})")
print(f"  E[r_P*]: {E_rp:.4f} ({E_rp:.2%})")
print(f"  sigma_P*: {std_rp:.4f} ({std_rp:.2%})")
print(f"  Sharpe Ratio: {sharpe_rp:.4f}\n")


rows = {}
for name in members:
    A_x = risk_aversion_df.loc["X", name]
    if math.isclose(A_x, 0, abs_tol=1e-9):  # risk-neutral: y* unbounded
        rows[name] = {"A_X": 0.0, "y*": math.inf,
                      "w_S": math.inf, "w_B": math.inf, "w_rf": -math.inf,
                      "w_S/w_B": w_rs / w_bf}
        continue
    y_star = (E_rp - rf) / (A_x * var_rp)
    rows[name] = {"A_X": A_x, "y*": y_star,
                  "w_S": y_star * w_rs, "w_B": y_star * w_bf,
                  "w_rf": 1 - y_star,
                  "w_S/w_B": (y_star * w_rs) / (y_star * w_bf)}

allocation_df = pd.DataFrame(rows).T   # rows = members
allocation_df.round(4)

Optimal Risky Portfolio (P*):
  w_S*: 0.4624 (46.24%)
  w_B*: 0.5376 (53.76%)
  E[r_P*]: 0.0785 (7.85%)
  sigma_P*: 0.1098 (10.98%)
  Sharpe Ratio: 0.4417



,A_X,y*,w_S,w_B,w_rf,w_S/w_B
erik,0.0,inf,inf,inf,-inf,0.86
elaine,3.0,1.341,0.6200,0.721,-0.341,0.86
chen,1.5,2.682,1.2401,1.442,-1.682,0.86


## Question 3
### Different Borrowing and Lending Rates

With a borrowing rate $r_{\text{borrow}} = 6\%$ above the lending rate $r_f = 3\%$, the capital allocation line has a kink at $y = 1$. The expected return of the complete portfolio $\text{C}$ becomes

$$E[r_C] = \begin{cases} r_f + y\left(E[r_{P^*}] - r_f\right), & y \le 1 \quad \text{(lending)} \\ r_{\text{borrow}} + y\left(E[r_{P^*}] - r_{\text{borrow}}\right), & y > 1 \quad \text{(borrowing)} \end{cases}$$

The borrowing cost only applies to the borrowed amount $(y-1)$, and $\sigma_C = y\,\sigma_{P^*}$ is unchanged. Beyond the kink the CAL is flatter, with slope $\left(E[r_{P^*}] - r_{\text{borrow}}\right)/\sigma_{P^*}$.

Applying the FOC on each segment gives three cases:

1. $y_L = \dfrac{E[r_{P^*}] - r_f}{A\sigma_{P^*}^2} \le 1$: the investor lends and is unaffected.
2. $y_B = \dfrac{E[r_{P^*}] - r_{\text{borrow}}}{A\sigma_{P^*}^2} \ge 1$: the investor borrows at $r_{\text{borrow}}$, and $y^* = y_B$.
3. $y_B < 1 < y_L$: the investor wants to borrow at $r_f$ but not at $r_{\text{borrow}}$, so $y^* = 1$.

Case 3 gives the range of $A$ for which the investor holds exactly 100% in $P^*$:

$$\frac{E[r_{P^*}] - r_{\text{borrow}}}{\sigma_{P^*}^2} \le A \le \frac{E[r_{P^*}] - r_f}{\sigma_{P^*}^2}$$

Utility is evaluated with the rate of the segment the investor ends up on: $U = E[r_C] - \tfrac{1}{2}A y^2 \sigma_{P^*}^2$.

In [4]:
r_borrow = 0.06  # borrowing rate; lending still earns rf

def optimal_y(A, E_rp, var_rp, r_lend, r_borrow):
    """Optimal y on the kinked CAL.

    Returns (y*, rate earned or paid on the risk-free position).
    """
    if math.isclose(A, 0, abs_tol=1e-9):
        # risk-neutral: borrows without limit while E[r_P*] > r_borrow
        return math.inf, r_borrow
    y_lend = (E_rp - r_lend) / (A * var_rp)
    if y_lend <= 1:
        return y_lend, r_lend  # lends (or exactly 100% in P*): unaffected
    y_borrow = (E_rp - r_borrow) / (A * var_rp)
    if y_borrow >= 1:
        return y_borrow, r_borrow  # borrows at 6%
    return 1.0, r_lend  # stuck at the kink: neither borrows nor lends

def utility(A, y, r, E_rp, var_rp):
    return r + y * (E_rp - r) - 0.5 * A * y ** 2 * var_rp

rows = {}
for name in members:
    A_x = risk_aversion_df.loc["X", name]
    if math.isclose(A_x, 0, abs_tol=1e-9):
        rows[name] = {"A_X": 0.0,
                      "y* (old)": math.inf, "y* (new)": math.inf,
                      "w_S": math.inf, "w_B": math.inf, "w_rf": -math.inf,
                      "U (old)": math.inf, "U (new)": math.inf}
        continue
    y_old = (E_rp - rf) / (A_x * var_rp)
    y_new, r = optimal_y(A_x, E_rp, var_rp, rf, r_borrow)
    rows[name] = {"A_X": A_x, "y* (old)": y_old, "y* (new)": y_new,
                  "w_S": y_new * w_rs, "w_B": y_new * w_bf,
                  "w_rf": 1 - y_new,
                  "U (old)": utility(A_x, y_old, rf, E_rp, var_rp),
                  "U (new)": utility(A_x, y_new, r, E_rp, var_rp)}

borrowing_df = pd.DataFrame(rows).T  # rows = members
print(borrowing_df.round(4))

# Range of A for which y* = 1 (neither borrowing nor lending)
A_low = (E_rp - r_borrow) / var_rp
A_high = (E_rp - rf) / var_rp
print(f"\nInvestor holds exactly 100% in P* for "
      f"{A_low:.4f} <= A <= {A_high:.4f}")

        A_X  y* (old)  y* (new)     w_S     w_B    w_rf  U (old)  U (new)
erik    0.0       inf       inf     inf     inf    -inf      inf      inf
elaine  3.0     1.341    1.0000  0.4624  0.5376  0.0000   0.0625   0.0604
chen    1.5     2.682    1.0228  0.4729  0.5499 -0.0228   0.0950   0.0695

Investor holds exactly 100% in P* for 1.5343 <= A <= 4.0230


## Question 4

**Is the advice consistent with (b)?** No. In (b) every member holds stocks and bonds in the same ratio, $w_S/w_B = 0.4624/0.5376 = 0.86$, and risk aversion only changes $y^*$, the split between $P^*$ and the risk-free asset. This is the separation property: under mean-variance utility with a single rate for borrowing and lending, a more risk-averse investor should hold more cash, not more bonds. The advisers' rule contradicts this.

**Reason 1: Borrowing costs more than lending (supported by our numbers in (c)).**
Separation needs investors to borrow and lend at the same rate. With $r_{\text{borrow}} = 6\%$, leverage on $P^*$ only earns $7.85\% - 6\% = 1.85\%$ per unit, and the slope of the CAL beyond $P^*$ drops from 0.44 to 0.17. An investor who wants more risk than $P^*$ can instead get it without borrowing, by tilting the risky portfolio towards stocks. For our group:

| Member | A | P* fixed: S / B / rf | U | Best mix: S / B / rf | U |
|---|---|---|---|---|---|
| Elaine | 3 | 46% / 54% / 0% | 0.0604 | 56% / 44% / 0% | 0.0609 |
| Chen | 1.5 | 47% / 55% / −2% | 0.0695 | 96% / 4% / 0% | 0.0757 |

Both are better off leaving $P^*$, and the optimal mix depends on $A$: the less risk-averse Chen holds almost only stocks, while Elaine holds more bonds. Investors with $A \geq 4.02$, who lend, keep $P^*$. In this setting the advisers' rule is rational.

**Reason 2: Preferences depend on more than mean and variance (supported by our numbers in (a)).**
Bets Y and Z have the same mean (7%) and the same standard deviation (16%). Under mean-variance utility, each member should therefore give them the same CE. None of us did:

| | CE(Y) | CE(Z) | $A_Y$ | $A_Z$ |
|---|---|---|---|---|
| Erik | 12% | 5% | −3.91 | 1.56 |
| Elaine | 3% | 3.5% | 3.12 | 2.73 |
| Chen | 5% | 2% | 1.56 | 3.91 |

The bets differ only in skewness. Y has a small chance of a large loss (negative skew), and Z has a small chance of a large gain (positive skew). Our answers show that we care about this third moment, and our implied $A$ is not even constant across bets. $P^*$ is optimal only for an investor who looks at mean and variance alone. An investor who dislikes left-tail risk, as Elaine does by valuing Z above Y, would rationally hold fewer of the assets with crash risk (typically stocks) and more bonds than in $P^*$.

*Other reasons (not required):* hedging of future liabilities or a long horizon, for which long-term bonds are close to risk-free; correlation with labour income; and estimation error in $E[r]$, which makes the extreme weights from mean-variance optimisation unreliable.